# Subsample datasets to 100 per year

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import random
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [ ]:
# Set up directories

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/complete_human/"

references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

# os.chdir(references)
# states_ref = pd.read_csv("states_ref.csv")

## Upload FASTAs

In [ ]:
# Organize fastas

fastas = {}
for dirpath, dirs, files in os.walk(home):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = df_from_fasta(file_name)
            fastas[file_name.split("/")[-1]] = fasta
    break 

In [9]:
print(fastas)

{'human_euro_H3_01-01-2000--12-31-2025.fasta':                                               full_header  \
0       >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
1       >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
2       >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
3       >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
4       >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
...                                                   ...   
423023  >EPI_ISL_163508|A/Pays_de_Loire/1262/2014|H3N2...   
423024  >EPI_ISL_163507|A/Pays_de_Loire/1266/2014|H3N2...   
423025  >EPI_ISL_163506|A/Caen/349/2014|H3N2|Caen|2014...   
423026  >EPI_ISL_163506|A/Caen/349/2014|H3N2|Caen|2014...   
423027  >EPI_ISL_163506|A/Caen/349/2014|H3N2|Caen|2014...   

                                                 sequence  
0       caggcaaaccatttgaatggatgtcaatccgactctactgttctta...  
1       agcaaaagcagggtgacaaagacataatggattccaacactgtgtc...  
2       atgaagactatcattgctttgagctacattct

## Subsample

In [19]:
# Create a dictionary of dictionaries of dataframes grouped by year

fastas_grouped = {} # Overall dictionary -- length is number of datasets needed

for key in fastas:
    years = {} # For each dataset, there are a number of years
    fasta = fastas[key]
    fasta["Year"] = fasta["full_header"].apply(lambda x: dateutil.parser.parse(x.split("|")[-2]).year) # Find year
    for year, rows in fasta.groupby("Year"): # Separate dataframe into multiple dataframes by year
        years[year] = rows # For each year, there are a number of entries that have that year
    fastas_grouped[key] = years

print(fastas_grouped)

{'human_euro_H3_01-01-2000--12-31-2025.fasta': {2000:                                               full_header  \
393244  >EPI_ISL_1908|A/Madrid/SO2913/00|H3N2|Madrid|2...   
393245  >EPI_ISL_1893|A/Salamanca/RR682/00|H3N2|Salama...   
393246  >EPI_ISL_1892|A/Zaragoza/RR658/00|H3N2|Zaragoz...   
393247  >EPI_ISL_1891|A/Zaragoza/RR653/00|H3N2|Zaragoz...   
393248  >EPI_ISL_1890|A/Madrid/RR610/00|H3N2|Madrid|20...   
...                                                   ...   
422392  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   
422393  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   
422394  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   
422395  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   
422396  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   

                                                 sequence  Year  
393244  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  2000  
393245  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  2000  
393246  gcctcat

In [ ]:
subsampled_dfs = {} # Overall subsampled dictionary -- length is number of datasets needed
for key in fastas_grouped:
    dataset = fastas_grouped[key] # Dataset
    df = pd.DataFrame() # Hold subsampled data
    for year_key in dataset: # Dictionary of years and their dataframes 
        year_df = dataset[year_key] # One year and its data
        # If there are more than 100 entries, subsample a random 100 
        subsampled = year_df[["full_header", "sequence"]].sample(n=100, random_state=2009) if len(year_df) > 100 else year_df[["full_header", "sequence"]]
        # print(subsampled)
        df = pd.concat([df, subsampled]) # Add subsampled data to dataframe
    subsampled_dfs[key] = df # Add dataframe to dictionary of datasets

print(subsampled_dfs)
        

                                              full_header  \
406185  >EPI_ISL_12723|A/Sachsen-Anhalt/6/00|H3N2|Sach...   
395512  >EPI_ISL_2685|A/Lyon/1242/2000|H3N2|Lyon|2000|...   
422389  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   
395441  >EPI_ISL_4083|A/France/5/00|H3N2|France|2000|h...   
395403  >EPI_ISL_4121|A/France/43/00|H3N2|France|2000|...   
...                                                   ...   
406202  >EPI_ISL_12721|A/Niedersachsen/56/00|H3N2|Nied...   
410044  >EPI_ISL_16036|A/Denmark/207/2000|H3N2|Denmark...   
395429  >EPI_ISL_4095|A/France/17/00|H3N2|France|2000|...   
395402  >EPI_ISL_4122|A/France/44/00|H3N2|France|2000|...   
409822  >EPI_ISL_115621|A/Netherlands/3/2000|H3N2|Neth...   

                                                 sequence  
406185  ttggaggtatgtttcatgtattcagattttcatttcatcaatgaac...  
395512  gcaaaagcaggagtgaaaatgaatccaaatcaaaagataataacga...  
422389  aaccatgaagactatcattgctttgagctacattttatgtctggtt...  
395441  caaaaacttcccggaaatg

## Download FASTAs

In [25]:
# Prepare for download
for key in subsampled_dfs:
    file_name = "subsampled_" + key # Create file name
    fasta = subsampled_dfs[key]
    df_to_fasta(fasta, file_name, home)
